# RiverScope geometry: distance-map quality control

Run `scripts/prepare_riverscope.py` first. This notebook inspects a derived tile alongside its original multiclass label; it does not edit either dataset.

In [ ]:
from pathlib import Path
import csv
import logging
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import tifffile

logging.getLogger('tifffile').setLevel(logging.ERROR)

from reservoir_segmentation.dataset import (
    resolve_riverscope_member,
    resolve_riverscope_path,
)

SOURCE = Path('../data/raw/RiverScope.zip')
PROCESSED_ROOT = Path('../data/processed/riverscope/geometry-v2')
SPLIT = 'train'
SIGMA = 3.0
ALPHA = 4.0

manifest_path = PROCESSED_ROOT / 'manifests' / f'{SPLIT}.csv'
if not SOURCE.exists() or not manifest_path.is_file():
    raise FileNotFoundError('Set SOURCE and PROCESSED_ROOT to prepared local data.')

with manifest_path.open(newline='', encoding='utf-8') as handle:
    rows = list(csv.DictReader(handle))
if not rows:
    raise ValueError(f'No retained rows in {manifest_path}')

row = rows[0]
print(f"Inspecting {row['normalized_planetscope_path']} ({len(rows)} retained {SPLIT} tiles)")

In [ ]:
def load_derived(column):
    return np.load(PROCESSED_ROOT / row[column])

def load_source_label(reference):
    if SOURCE.is_file() and SOURCE.suffix.lower() == '.zip':
        with zipfile.ZipFile(SOURCE) as archive:
            member = resolve_riverscope_member(reference, archive.namelist())
            with archive.open(member) as handle, tifffile.TiffFile(handle) as tif:
                return tif.asarray()
    return tifffile.imread(resolve_riverscope_path(SOURCE, reference))

labels = load_source_label(row['label_path'])
binary = load_derived('binary_mask_path')
valid = load_derived('valid_mask_path')
boundary = load_derived('boundary_path')
distance_squared = load_derived('distance_squared_path')
weights = 1.0 + ALPHA * np.exp(-distance_squared / (2.0 * SIGMA**2))

assert labels.shape == binary.shape == valid.shape == boundary.shape == distance_squared.shape
print('shape:', labels.shape, '| valid:', valid.mean(), '| shoreline pixels:', boundary.sum())

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True)
plots = [
    (labels, 'Original label (0 / 1 / 2)', 'tab20'),
    (binary, 'Derived binary water mask', 'Blues'),
    (valid, 'Validity map', 'gray'),
    (boundary, 'Thick 8-neighbour shoreline', 'magma'),
    (distance_squared, 'Squared distance to shoreline', 'viridis'),
    (weights, f'Weight map (sigma={SIGMA}, alpha={ALPHA})', 'viridis'),
]
for axis, (array, title, cmap) in zip(axes.flat, plots):
    image = axis.imshow(array, cmap=cmap)
    axis.set_title(title)
    axis.axis('off')
    fig.colorbar(image, ax=axis, shrink=0.72)
plt.show()

Inspect diagonal shorelines and the tile frame at high zoom. A coastline must only appear where two **valid** labels disagree; neither the outer tile edge nor a transition to invalid data is a coast. `distance_squared` is the cacheable geometric source of truth, while the final weight map changes with `SIGMA` and `ALPHA`.